# 070 — Fusión multimodal y representación conjunta

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Tres estrategias de fusión:** **temprana** (concatenar señales/características y entrenar
un modelo único: interacciones ricas, frágil ante modalidades faltantes), **tardía** (un
modelo por modalidad y combinar decisiones — promedio, producto, votación: modular y
robusta, pero ciega a interacciones) e **intermedia** (codificadores separados que se
comunican dentro de la red, típicamente con **atención cruzada** — el estándar actual).

**Atención cruzada:** Q sale de la modalidad A; K y V de la B:
`Atención(Q,K,V) = softmax(Q·Kᵀ/√d)·V`. Cada elemento de A pregunta qué partes de B le son
relevantes y recibe un resumen ponderado. La dirección importa (texto→imagen ≠
imagen→texto). Flamingo inserta estas capas en un LLM congelado.

**Alineación de espacios:** contrastiva (CLIP), proyección aprendida al espacio de tokens
del LLM (LLaVA), CCA como precursor clásico. También hay que alinear tiempo y granularidad.

**Problemas prácticos:** modalidades faltantes (la tardía degrada con gracia; la temprana
necesita *dropout de modalidades*) y **dominancia** (una modalidad predictiva hace que el
modelo ignore la otra — se diagnostica con ablaciones).


### 🧮 Cálculo de referencia (para los ejercicios)

```text
Q=(2,0); K₁=(1,0),V₁=(1,0); K₂=(0,1),V₂=(0,1); d=2
puntajes: 2/√2 = 1.414 y 0 → softmax (0.80, 0.20)
salida: 0.80·V₁ + 0.20·V₂ = (0.80, 0.20)
```


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("attention", seed=70)
show(result)


## Reflexión

1. Tu modelo audiovisual rinde casi igual cuando anulas por completo el audio. ¿Qué
   fenómeno es, con qué experimento lo confirmas y qué cambiarías en el entrenamiento?
2. ¿Por qué el sarcasmo (texto positivo + tono de voz negativo) es indetectable para una
   fusión tardía **por diseño**, y cuál es la modificación mínima que lo haría detectable?
3. En Flamingo la atención cruzada usa Q del texto y K,V de la imagen. ¿Qué cambiaría
   conceptualmente si fuera al revés, y por qué esa dirección es la adecuada cuando el
   objetivo es generar texto?
